In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from collections import deque
from model import GraphSAGE, LinkPredictor
from utils import TaskBPipeline
np.random.seed(42)

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = torch.load('models/task_a_model_new.pt', weights_only=False)
data = checkpoint['data']
student_id_map = checkpoint['student_id_map']
course_id_map = checkpoint['course_id_map']
concept_id_map = checkpoint['concept_id_map']

model = GraphSAGE(hidden_dim=256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load LinkPredictor from Task B
link_model = LinkPredictor(emb_dim=256)
link_model.load_state_dict(torch.load('models/task_b_link_model_new.pt', weights_only=True))
link_model.eval()

# Get concept embeddings
with torch.no_grad():
    out = model(data)
    concept_embs = out[2]

idx_to_concept = {v: k for k, v in concept_id_map.items()}

# Load datasets
enrollments = pd.read_csv('datasets/enrollments.csv')
courses = pd.read_csv('datasets/courses.csv')
chatbot = pd.read_csv('datasets/chatbot_signals.csv')
assessments = pd.read_csv('datasets/assessment_scores.csv')
course_concepts_df = pd.read_csv('datasets/course_concepts.csv')
concepts = pd.read_csv('datasets/concepts.csv')

course_name_map = courses.set_index('course_id')['course_name'].to_dict()
concept_name_map = concepts.set_index('concept_id')['concept_name'].to_dict()

# Initialize Task B pipeline
pipeline = TaskBPipeline(model, link_model, data, concept_id_map,
                         idx_to_concept, concept_embs, chatbot,
                         course_concepts_df, concept_name_map)

print("All models and pipeline loaded!")

All models and pipeline loaded!


In [3]:
course_name_map = courses.set_index('course_id')['course_name'].to_dict()
concept_name_map = concepts.set_index('concept_id')['concept_name'].to_dict()

failing = enrollments[enrollments['grade'] < 1.5]

course_stats = enrollments.groupby('course_id').agg(
    total_enrolled = ('student_id', 'count'),
    avg_grade = ('grade', 'mean'),
    fail_count = ('passed', lambda x: (x==0).sum()),
    fail_rate = ('passed', lambda x: (x==0).mean()),
).reset_index()

course_stats['course_name'] = course_stats['course_id'].map(course_name_map)
course_stats = course_stats.sort_values('fail_rate', ascending = False)

print("Top 10 courses by failure rate: ")
print(course_stats[['course_name', 'total_enrolled', 'fail_count', 'fail_rate', 'avg_grade']].head(10).to_string(index = False))

Top 10 courses by failure rate: 
                 course_name  total_enrolled  fail_count  fail_rate  avg_grade
         Full Stack Capstone             541          72   0.133087   2.012957
Product Engineering Capstone             501          61   0.121756   2.024291
              Cloud Capstone             484          57   0.117769   2.031942
   Data Engineering Capstone             476          55   0.115546   1.995861
   Research Methods Capstone             528          61   0.115530   2.015928
            Systems Capstone             495          54   0.109091   2.046667
               Deep Learning             746          81   0.108579   2.093794
  Cybersecurity Fundamentals             760          81   0.106579   2.079789
      Cybersecurity Capstone             472          50   0.105932   2.038157
        Software Engineering             768          81   0.105469   2.077318


In [4]:
failing_students = failing['student_id'].unique()

failing_chatbot = chatbot[chatbot['student_id'].isin(failing_students)]

concept_to_course = course_concepts_df.groupby('concept_id')['course_id'].apply(list).to_dict()

results = []
for course_id in course_stats['course_id']:
    course_concepts = course_concepts_df[course_concepts_df['course_id'] == course_id]['concept_id'].tolist()

    course_failing = failing[failing['course_id'] == course_id]['student_id'].tolist()

    relevant = chatbot[
        (chatbot['student_id'].isin(course_failing)) &
        (chatbot['concept_id'].isin(course_concepts))
    ]

    if len(relevant) > 0:
        concept_confusion = relevant.groupby('concept_id')['confusion_score'].mean().sort_values(ascending=False)
        
        for concept_id, confusion in concept_confusion.head(3).items():
            results.append({
                'course': course_name_map[course_id],
                'concept': concept_name_map[concept_id], 
                'avg_confusion': round(confusion, 3)
            })

results_df = pd.DataFrame(results)
print("Top 3 most confusing concepts per struggling course:")
print(results_df.to_string(index=False))

Top 3 most confusing concepts per struggling course:
                      course                        concept  avg_confusion
         Full Stack Capstone               Model Deployment          0.470
         Full Stack Capstone            APIs and Interfaces          0.469
         Full Stack Capstone                  Microservices          0.435
Product Engineering Capstone               Model Deployment          0.482
Product Engineering Capstone              Usability Testing          0.279
Product Engineering Capstone                    Prototyping          0.264
              Cloud Capstone                 Cloud Services          0.625
              Cloud Capstone                Fault Tolerance          0.545
              Cloud Capstone               Containerization          0.544
   Data Engineering Capstone                 Data Pipelines          0.621
   Data Engineering Capstone                 Cloud Services          0.592
   Data Engineering Capstone               Data

In [5]:
model.eval()
student_indices = torch.tensor([student_id_map[s] for s in enrollments['student_id']], dtype = torch.long)
course_indices = torch.tensor([course_id_map[c] for c in enrollments['course_id']], dtype = torch.long)

with torch.no_grad():
    out = model(data)
    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]
    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs], dim = 1)
    baseline_preds = model.task_a_head(pairs).argmax(dim = 1)

print(f"Baseline total High risk: {(baseline_preds == 2).sum().item()}")

Baseline total High risk: 6293


In [6]:
confusion_lookup = chatbot.set_index(['student_id','concept_id'])['confusion_score'].to_dict()
print(f"Total student-concept pairs in lookup: {len(confusion_lookup)}")

Total student-concept pairs in lookup: 35006


In [7]:
def simulate_concept_tutoring(course_id, reduction = 0.5):
    course_name = course_name_map[course_id]
    course_concepts = course_concepts_df[course_concepts_df['course_id'] == course_id]['concept_id'].tolist()

    course_enrollments = enrollments[enrollments['course_id'] == course_id]
    course_students = course_enrollments['student_id'].tolist()

    course_mask = (enrollments['course_id'] == course_id).values
    baseline_count = (baseline_preds[course_mask] == 2).sum().item()

    concept_results = []

    for concept_id in course_concepts:
        affected_students = []
        for student_id in course_students:
            conf = confusion_lookup.get((student_id, concept_id), None)
            if conf is not None:
                affected_students.append(student_id_map[student_id])

        if len(affected_students) ==0:
            continue

        original_student_x = data['student'].x.clone()

        for s_idx in affected_students:
            data['student'].x[s_idx, 3] = data['student'].x[s_idx, 3] * reduction

        with torch.no_grad():
            out = model(data)
            pairs = torch.cat([out[0][student_indices], out[1][course_indices],
                               out[0][student_indices] - out[1][course_indices],
                               out[0][student_indices] * out[1][course_indices]], dim=1)
            new_preds = model.task_a_head(pairs).argmax(dim = 1)

        new_count = (new_preds[course_mask] ==2 ).sum().item()
        helped = baseline_count - new_count

        concept_results.append({
            'concept': concept_name_map[concept_id],
            'students_affected': len(affected_students),
            'students_helped': helped
        })

        data['student'].x = original_student_x

    concept_results.sort(key = lambda x: x['students_helped'], reverse=True)

    return course_name, baseline_count, concept_results

In [8]:
def get_systemic_gaps(course_id, top_n=5):
    course_name = course_name_map[course_id]
    
    high_risk_students = enrollments[
        (enrollments['course_id'] == course_id) &
        (enrollments['risk_class'] == 'High')
    ]['student_id'].tolist()
    
    if len(high_risk_students) == 0:
        print(f"No high-risk students for {course_name}")
        return []
    
    concept_counts = {}
    concept_total_confusion = {}
    
    for student_id in high_risk_students:
        weak, no_data = pipeline.get_weak_concepts(student_id, course_id, top_k=3)
        
        for concept_id, confusion in weak:
            if concept_id not in concept_counts:
                concept_counts[concept_id] = 0
                concept_total_confusion[concept_id] = 0
            concept_counts[concept_id] += 1
            concept_total_confusion[concept_id] += confusion
    
    gaps = []
    for concept_id, count in concept_counts.items():
        avg_confusion = concept_total_confusion[concept_id] / count
        gaps.append({
            'concept_id': concept_id,
            'concept_name': concept_name_map[concept_id],
            'student_count': count,
            'pct_of_high_risk': round(count / len(high_risk_students) * 100, 1),
            'avg_confusion': round(avg_confusion, 3)
        })
    
    gaps.sort(key=lambda x: x['student_count'], reverse=True)
    
    print(f"\n{'='*60}")
    print(f"Systemic Gaps: {course_name}")
    print(f"High-risk students: {len(high_risk_students)}")
    print(f"{'='*60}")
    
    for i, gap in enumerate(gaps[:top_n], 1):
        print(f"  {i}. {gap['concept_name']} [{gap['concept_id']}]")
        print(f"     Students struggling: {gap['student_count']}/{len(high_risk_students)} ({gap['pct_of_high_risk']}%)")
        print(f"     Avg confusion: {gap['avg_confusion']}")
    
    print(f"{'='*60}\n")
    return gaps[:top_n]

In [9]:
print("=== Task B → Task D: Systemic Gap Analysis + Validation ===\n")

for course_id in course_stats['course_id'].head(5):
    gaps = get_systemic_gaps(course_id)
    
    if gaps:
        course_name, baseline, sim_results = simulate_concept_tutoring(course_id)
        
        print(f"  Simulation validation ({course_name}):")
        print(f"  Baseline high-risk: {baseline}")
        for result in sim_results[:3]:
            print(f"    Tutoring on {result['concept']}: {result['students_helped']} students helped")
        print()

=== Task B → Task D: Systemic Gap Analysis + Validation ===


Systemic Gaps: Full Stack Capstone
High-risk students: 72
  1. Model Deployment [K043]
     Students struggling: 19/72 (26.4%)
     Avg confusion: 0.47
  2. Concurrency and Threads [K011]
     Students struggling: 18/72 (25.0%)
     Avg confusion: 0.353
  3. Design Systems [K080]
     Students struggling: 15/72 (20.8%)
     Avg confusion: 0.286
  4. Microservices [K057]
     Students struggling: 14/72 (19.4%)
     Avg confusion: 0.436
  5. Testing and Validation [K010]
     Students struggling: 14/72 (19.4%)
     Avg confusion: 0.445

  Simulation validation (Full Stack Capstone):
  Baseline high-risk: 119
    Tutoring on Testing and Validation: 13 students helped
    Tutoring on Microservices: 12 students helped
    Tutoring on Model Deployment: 10 students helped


Systemic Gaps: Product Engineering Capstone
High-risk students: 61
  1. Usability Testing [K079]
     Students struggling: 18/61 (29.5%)
     Avg confusion: 0.2